# Lesson 01 — Video Tokens, Attention, and the ViT Block

**Read first:** `docs/02_vision_transformers.md`

By the end of this notebook you will have built, from raw tensor operations:
1. the tubelet embedding that turns a video into tokens,
2. multi-head self-attention (and verified it against PyTorch's fused kernel),
3. 3D sin-cos positional embeddings (and *seen* their structure),
4. a full transformer block.

These are the exact components `vjepa_mini` uses — you're building your own tools before using them.

**Rule of engagement:** every exercise cell asks you to predict the output *before* running it. Write your prediction in the empty markdown cell above the code. Wrong predictions are where learning happens.

In [ ]:
#@title Setup — run this first { display-mode: "form" }
# Works both on Colab (clones the repo) and locally (repo already present).
import os, sys

if os.path.exists("../src/vjepa_mini"):          # running locally from notebooks/
    sys.path.insert(0, os.path.abspath("../src"))
elif os.path.exists("world-model-jepa/src"):      # already cloned on Colab
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))
else:                                             # fresh Colab runtime
    REPO_URL = "https://github.com/josephlionel95-moon/world-model-jepa.git"
    os.system(f"git clone {REPO_URL}")
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("NOTE: Runtime > Change runtime type > T4 GPU for the training notebooks.")


## Part 1 — A video becomes tokens

A transformer consumes a sequence of vectors. V-JEPA converts a clip `[C, T, H, W]` into tokens by cutting it into **tubelets** — space-time boxes of shape `(2, 8, 8)` in our mini config — and linearly projecting each one to a `D`-dim vector.

**Claim to verify:** a `Conv3d` with `kernel_size == stride` is *exactly* the same as flattening each tubelet and applying one shared `nn.Linear`. The conv is just the fast way to write it.

In [ ]:
import torch
import torch.nn as nn

B, C, T, H, W = 2, 1, 8, 64, 64
tt, p, D = 2, 8, 192           # tubelet time, patch size, embed dim
video = torch.randn(B, C, T, H, W)

# --- the conv version (what the library uses) ---
conv = nn.Conv3d(C, D, kernel_size=(tt, p, p), stride=(tt, p, p))
tokens_conv = conv(video).flatten(2).transpose(1, 2)     # [B, N, D]

# --- the "obvious" version: unfold into tubelets, then one Linear ---
# unfold carves out non-overlapping windows along each dim
tub = video.unfold(2, tt, tt).unfold(3, p, p).unfold(4, p, p)
# tub: [B, C, T', H', W', tt, p, p]  -> flatten each tubelet's pixels
tub = tub.permute(0, 2, 3, 4, 1, 5, 6, 7).reshape(B, -1, C * tt * p * p)
linear = nn.Linear(C * tt * p * p, D)
# copy the conv's weights into the linear layer so they compute the same map
linear.weight.data = conv.weight.data.reshape(D, -1)
linear.bias.data = conv.bias.data
tokens_lin = linear(tub)

print("token grid:", tokens_conv.shape)   # expect [2, 256, 192]  (4*8*8 = 256)
print("max abs difference:", (tokens_conv - tokens_lin).abs().max().item())
assert (tokens_conv - tokens_lin).abs().max() < 1e-4
print("=> a patch embed IS a linear layer per tubelet. No magic.")

**Question (answer before moving on):** each of our tokens spans 2 frames. What information does a 2-frame token carry that a 1-frame token cannot? Why might that matter for a model that must understand *motion*?

## Part 2 — Attention from scratch

Each token emits a **query** ("what am I looking for?"), a **key** ("what do I contain?") and a **value** ("what do I offer?"). Token i gathers information from all tokens j with weights `softmax_j(q_i . k_j / sqrt(d))`.

Implement it with explicit matmuls, then check against `F.scaled_dot_product_attention`.

In [ ]:
import torch.nn.functional as F

def attention_from_scratch(x, num_heads):
    """x: [B, N, D] -> [B, N, D]. Weights are identity here (pure mechanism)."""
    B, N, Dm = x.shape
    hd = Dm // num_heads
    # split into heads: [B, heads, N, hd]
    q = x.reshape(B, N, num_heads, hd).transpose(1, 2)
    k, v = q, q                      # self-attention with tied q=k=v for the demo
    scores = q @ k.transpose(-2, -1) / (hd ** 0.5)   # [B, heads, N, N]
    weights = scores.softmax(dim=-1)
    out = weights @ v                                 # [B, heads, N, hd]
    return out.transpose(1, 2).reshape(B, N, Dm), weights

x = torch.randn(2, 256, 192)
mine, attn_weights = attention_from_scratch(x, num_heads=6)

q = x.reshape(2, 256, 6, 32).transpose(1, 2)
ref = F.scaled_dot_product_attention(q, q, q).transpose(1, 2).reshape(2, 256, 192)
print("max abs diff vs fused kernel:", (mine - ref).abs().max().item())
assert (mine - ref).abs().max() < 1e-4

### Why the sqrt(d) matters

If q, k have d unit-variance components, `q.k` has variance d — std grows like sqrt(d). Softmax of large-magnitude scores saturates to one-hot, and gradients through a saturated softmax vanish.

**Predict:** what happens to the entropy of the attention weights if we *remove* the scaling at d=32? Higher or lower? Then run:

In [ ]:
import math

def attn_entropy(scale):
    q = torch.randn(1000, 32)
    k = torch.randn(1000, 32)
    w = (q @ k.T * scale).softmax(dim=-1)
    return -(w * (w + 1e-12).log()).sum(-1).mean().item()

print(f"with 1/sqrt(d) scaling : entropy = {attn_entropy(1/math.sqrt(32)):.3f}")
print(f"without scaling        : entropy = {attn_entropy(1.0):.3f}")
print(f"(uniform over 1000 keys would be {math.log(1000):.3f})")
# Lower entropy = closer to one-hot = saturated softmax = tiny gradients.

## Part 3 — Positional embeddings you can see

Attention is permutation-equivariant: it has no idea *where* a token is. V-JEPA adds fixed **3D sin-cos codes** (separate codes for t, h, w, concatenated). Let's look at their similarity structure — you should be able to *read off* the grid geometry.

In [ ]:
import matplotlib.pyplot as plt
from vjepa_mini.models.patch_embed import sincos_3d

pe = sincos_3d(dim=192, grid_t=4, grid_h=8, grid_w=8)     # [256, 192]
sim = torch.nn.functional.normalize(pe, dim=1) @ torch.nn.functional.normalize(pe, dim=1).T

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im = axes[0].imshow(sim)
axes[0].set_title("cosine similarity of all 256 position codes")
plt.colorbar(im, ax=axes[0])
# similarity of ONE position (t=1, h=3, w=4) to every position, shown per-frame
ref_idx = 1 * 64 + 3 * 8 + 4
grid = sim[ref_idx].reshape(4, 8, 8)
axes[1].imshow(torch.cat(list(grid), dim=1))
axes[1].set_title("one token's similarity to the whole grid (frames side by side)")
axes[1].set_axis_off()
plt.tight_layout(); plt.show()

# READ THE LEFT PLOT: the 64x64 blocks along the diagonal are frames
# (same-t positions are more similar); inside each block, 8-wide stripes
# are rows. The flattening order (t, h, w) is literally visible.
# RIGHT PLOT: similarity is highest at the token's own (h, w) in its own
# frame and decays with distance in ALL THREE axes - that's what gives the
# predictor a notion of "where".

**The classic silent bug:** if the token order from the patch embed were `(h, w, t)` but the position table used `(t, h, w)`, training would still run — attention doesn't care — but every positional code would describe the wrong location. The model would slowly learn around the damage and you'd lose accuracy without an error message. This plot is how you catch it: the block structure must match the flattening order you *intended*.

## Part 4 — The full block, and a mini encoder

A transformer block = communicate (attention) + compute (MLP), each wrapped in pre-norm + residual. The library's `Block` is exactly what you just built. Let's assemble the actual mini V-JEPA encoder and push a video through it.

In [ ]:
from vjepa_mini.config import VJEPAConfig
from vjepa_mini.models.vit import VideoViT

cfg = VJEPAConfig()
enc = VideoViT(dim=cfg.enc_dim, depth=cfg.enc_depth, num_heads=cfg.enc_heads)
video = torch.rand(2, 1, cfg.num_frames, cfg.img_size, cfg.img_size)

full = enc(video)
print("full encoding:   ", full.shape)          # [2, 256, 192]

# The masked path: keep only 32 of 256 tokens (~87% masked, like V-JEPA)
keep = torch.randperm(256)[:32]
ctx = enc(video, keep_idx=keep)
print("context encoding:", ctx.shape)            # [2, 32, 192]

n_params = sum(p.numel() for p in enc.parameters())
print(f"encoder params: {n_params/1e6:.2f}M  (paper's ViT-L: ~300M)")

Look at `VideoViT.forward` in `src/vjepa_mini/models/vit.py` right now and find the line where positions are added — **before** token dropping. Explain to yourself why the other order would break the predictor. (Answer in `docs/02`, section 2.6.)

## Exercises

1. **Tubelet ablation setup.** Change the demo in Part 1 to `tt=1` (single-frame tokens). How many tokens does a clip produce now? What does that do to attention FLOPs (formula, not code)?
2. **Entropy vs width.** Extend the Part 2 experiment: plot attention entropy without scaling as d goes 8 → 512. At what width does it get catastrophic?
3. **Break the ordering on purpose.** Build `sincos_3d` with `(h, w, t)` ordering and redraw the Part 3 plots. Describe how the structure changes — this is the fingerprint of the bug.
4. **Challenge.** Add a causal mask to your from-scratch attention (each token attends only to earlier *frames*, full attention within a frame). You will need exactly this if you later try the paper's "causal multi-block" ablation.

## Check yourself

- Why does masking 90% of tokens cut encoder attention cost ~100x rather than ~10x?
- The MLP has 2/3 of the block's parameters. What job does it do that attention cannot?

Next: `02_masked_autoencoder_mini.ipynb` — your first self-supervised model, and a lesson in why pixel targets disappoint.